# Milestone 3 — RAG Pipeline (Retrieval, Reranking, and Adversarial Context)

This notebook builds a FAISS knowledge base from `train.csv`, then walks through the two-stage retrieval pipeline (bi-encoder retrieval + cross-encoder reranking), and answers Q1–Q8.

**Before running:** make sure `train.csv` (and `test.csv` if you need it) are in the working directory, or update the `pd.read_csv(...)` path below to point at your dataset (e.g. `/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv`).

## Setup — install dependencies

In [ ]:
!pip install -q faiss-cpu sentence-transformers transformers

## Setup — build the knowledge base and FAISS index

The knowledge base `kb` is built by taking, for every row of `train.csv`, the text of the *correct* option only (i.e. `row[row['answer']]`). Each entry in `kb` therefore sits at the same index as its source row in `train`. We embed every entry with `all-MiniLM-L6-v2` and load the embeddings into a flat (exact, L2) FAISS index.

In [ ]:
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("train.csv")

print("Creating knowledge base")
kb = []
for idx, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating index")
model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = model.encode(kb, show_progress_bar=False)
index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created:", len(kb), "entries")

## Setup — zero-shot classifier and the row-150 test case (used in Q1, Q2, Q6)

In [ ]:
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = [str(row_150["A"]), str(row_150["B"]), str(row_150["C"]), str(row_150["D"]), str(row_150["E"])]
ans_150 = str(row_150[row_150["answer"]])

print("Prompt:", prompt_150)
print("Correct letter:", row_150["answer"])

---
## Q1 — Zero-shot classification on the raw prompt (row 150)

Run `facebook/bart-large-mnli` on `prompt_150` with the 5 option texts as `candidate_labels`. Report the predicted probability of the ground-truth option (rounded to 3 dp).

In [ ]:
result_150 = zs(prompt_150, candidate_labels=labels_150)
score_dict_150 = dict(zip(result_150["labels"], result_150["scores"]))

q1_answer = round(score_dict_150[ans_150], 3)
print("Q1: probability assigned to the correct option =", q1_answer)

---
## Q2 — FAISS retrieval rank for row 150

Embed `prompt_150`, query the FAISS index for the top `k=10` nearest KB entries, and find the rank (1-indexed) at which the true document — the KB entry originally built from row 150 — appears.

In [ ]:
query_emb_150 = model.encode([prompt_150])
D_150, I_150 = index.search(np.array(query_emb_150), k=10)
retrieved_indices_150 = I_150[0].tolist()

print("Top-10 retrieved KB indices:", retrieved_indices_150)

if 150 in retrieved_indices_150:
    q2_answer = retrieved_indices_150.index(150) + 1
else:
    q2_answer = None  # not found in the top 10

print("Q2: rank of the true document =", q2_answer)

---
## Q3 — Cross-encoder reranking of the top-10 (row 150)

Load `cross-encoder/ms-marco-MiniLM-L-6-v2`, score `prompt_150` against each of the 10 documents retrieved above, sort by score, and find the new rank of the true document.

In [ ]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

docs_10 = [kb[i] for i in retrieved_indices_150]           
pairs = [[prompt_150, doc] for doc in docs_10]              
ce_scores = cross_encoder.predict(pairs)                   

ranked = sorted(zip(retrieved_indices_150, ce_scores), key=lambda x: x[1], reverse=True)
ranked_indices = [idx for idx, score in ranked]

print("Cross-encoder re-ranked KB indices:", ranked_indices)

if 150 in ranked_indices:
    q3_answer = ranked_indices.index(150) + 1
else:
    q3_answer = None

print("Q3: rank of the true document after reranking =", q3_answer)

---
## Q4 — Token count for the augmented string at row 42

Retrieve the top `k=5` documents for the prompt at row index 42, concatenate them with a single space, build `"Context: [concatenated_docs] Question: [prompt]"`, and count tokens with the `bert-base-uncased` tokenizer (no truncation).

In [ ]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

emb_42 = model.encode([prompt_42])
D_42, I_42 = index.search(np.array(emb_42), k=5)
docs_42 = [kb[i] for i in I_42[0]]

concatenated_docs_42 = " ".join(docs_42)
rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
tokens_42 = bert_tokenizer(rag_string_42, truncation=False)

q4_answer = len(tokens_42["input_ids"])   # includes [CLS]/[SEP] special tokens
print("Q4: total token count =", q4_answer)

---
## Q5 — Zero-shot with the TRUE document as context (row 150)

Use the exact true document for row 150 (`kb[150]`) as context, build the RAG string, and rerun the Q1 zero-shot classification on it.

In [ ]:
true_doc_150 = kb[150]
rag_string_150_true = f"Context: {true_doc_150} Question: {prompt_150}"

result_150_true = zs(rag_string_150_true, candidate_labels=labels_150)
score_dict_150_true = dict(zip(result_150_true["labels"], result_150_true["scores"]))

q5_answer = round(score_dict_150_true[ans_150], 3)
print("Q5: probability of the correct option with the TRUE context =", q5_answer)

---
## Q6 — Adversarial RAG: force an unrelated document as context (row 150)

Replace the context with the document at KB index 999 (a deliberately unrelated fact) and rerun zero-shot classification on `prompt_150`.

In [ ]:
adversarial_doc = kb[999]
rag_string_150_adv = f"Context: {adversarial_doc} Question: {prompt_150}"

result_150_adv = zs(rag_string_150_adv, candidate_labels=labels_150)
score_dict_150_adv = dict(zip(result_150_adv["labels"], result_150_adv["scores"]))

q6_answer = round(score_dict_150_adv[ans_150], 3)
print("Q6: probability of the correct option with ADVERSARIAL context =", q6_answer)

---
## Q7 — Retrieval Hit Rate over the first 100 rows

For rows 0–99, retrieve the top `k=5` documents per prompt. A row is a "hit" if the exact text of its correct option string appears verbatim inside at least one of the 5 retrieved documents.

In [ ]:
prompts_100 = [str(train.iloc[i]["prompt"]) for i in range(100)]
embs_100 = model.encode(prompts_100, show_progress_bar=False)
D_100, I_100 = index.search(np.array(embs_100), k=5)

hits = 0
for i in range(100):
    row = train.iloc[i]
    correct_text = str(row[row["answer"]])
    retrieved_docs = [kb[j] for j in I_100[i]]
    if any(correct_text in doc for doc in retrieved_docs):
        hits += 1

q7_answer = round(hits / 100 * 100, 1)
print(f"Q7: Hit Rate = {hits}/100 = {q7_answer}%")

---
## Q8 — Full RAG pipeline (retrieve → rerank → augment → predict → score) on rows 0–19

For each of the first 20 rows: retrieve top-5 with FAISS, rerank with the cross-encoder and keep the single best document, build the RAG string, classify with `bart-large-mnli` over the 5 options, rank the options by predicted probability, take the top 3, and compute MAP@3 against the true answer. Report the average MAP@3 across the 20 rows.

In [ ]:
def apk(actual_letter, predicted_letters, k=3):
    """Average precision @ k for a single row (only one relevant item: the true letter)."""
    predicted_letters = predicted_letters[:k]
    for i, letter in enumerate(predicted_letters):
        if letter == actual_letter:
            return 1.0 / (i + 1)
    return 0.0

options = ["A", "B", "C", "D", "E"]
map_scores = []

for i in range(20):
    row = train.iloc[i]
    prompt_i = str(row["prompt"])
    correct_letter = row["answer"]
    labels_i = [str(row[o]) for o in options]

    emb_i = model.encode([prompt_i])
    D_i, I_i = index.search(np.array(emb_i), k=5)
    docs_i = [kb[j] for j in I_i[0]]

    
    pairs_i = [[prompt_i, doc] for doc in docs_i]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc_i = docs_i[int(np.argmax(ce_scores_i))]


    rag_string_i = f"Context: {best_doc_i} Question: {prompt_i}"

   
    result_i = zs(rag_string_i, candidate_labels=labels_i)
    score_dict_i = dict(zip(result_i["labels"], result_i["scores"]))

    ranked_letters = sorted(options, key=lambda o: score_dict_i[str(row[o])], reverse=True)
    top3_letters = ranked_letters[:3]
    map_scores.append(apk(correct_letter, top3_letters, k=3))

q8_answer = round(float(np.mean(map_scores)), 3)
print("Per-row AP@3:", [round(s, 3) for s in map_scores])
print("Q8: average MAP@3 across 20 rows =", q8_answer)

---
## Summary of answers

In [ ]:
print("Q1 (zero-shot prob, row 150):        ", q1_answer)
print("Q2 (FAISS rank, row 150):             ", q2_answer)
print("Q3 (cross-encoder rank, row 150):     ", q3_answer)
print("Q4 (token count, row 42):             ", q4_answer)
print("Q5 (zero-shot prob w/ true ctx):      ", q5_answer)
print("Q6 (zero-shot prob w/ adversarial ctx):", q6_answer)
print("Q7 (Hit Rate %, rows 0-99):           ", q7_answer)
print("Q8 (avg MAP@3, rows 0-19):            ", q8_answer)